# Silver Data: Promotions + EPROM
Dùng promotions.csv làm nguồn chính. Kiểm tra từng lỗi của EPROM, đối chiếu các trường còn đúng với mã khuyến mãi gốc, rồi bỏ bản bổ sung đã có.
Cuối cùng so sánh toàn bộ thông tin; chỉ thêm mã gốc mới có dữ liệu hợp lệ. Khác thông tin trên cùng mã được giữ riêng để kiểm tra.


In [1]:
import pandas as pd
import numpy as np


## PHẦN 1: Đọc & Làm sạch Promotions (bảng master)

In [2]:
promotions = pd.read_csv('../promotions.csv')
display(promotions.head())
promotions.info()

,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
0,PROMO-0001,Spring Sale 2013,percentage,12.0,2013-03-18,2013-04-17,NaN,email,1,0
1,PROMO-0002,Mid-Year Sale 2013,percentage,18.0,2013-06-23,2013-07-22,NaN,online,0,0
2,PROMO-0003,Fall Launch 2013,percentage,10.0,2013-08-30,2013-10-02,NaN,email,0,0
3,PROMO-0004,Year-End Sale 2013,percentage,20.0,2013-11-18,2014-01-02,NaN,all_channels,0,50000
4,PROMO-0005,Urban Blowout 2013,fixed,50.0,2013-07-30,2013-09-02,Streetwear,online,0,150000


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   promo_id             50 non-null     object 
 1   promo_name           50 non-null     object 
 2   promo_type           50 non-null     object 
 3   discount_value       50 non-null     float64
 4   start_date           50 non-null     object 
 5   end_date             50 non-null     object 
 6   applicable_category  10 non-null     object 
 7   promo_channel        50 non-null     object 
 8   stackable_flag       50 non-null     int64  
 9   min_order_value      50 non-null     int64  
dtypes: float64(1), int64(2), object(7)
memory usage: 4.0+ KB


In [3]:
# Kiểm tra null từng cột
promotions.isnull().sum()

promo_id                0
promo_name              0
promo_type              0
discount_value          0
start_date              0
end_date                0
applicable_category    40
promo_channel           0
stackable_flag          0
min_order_value         0
dtype: int64

In [4]:
# Theo Dictionary, promo_id là PK => kiểm tra trùng lặp
print('promo_id là unique:', promotions['promo_id'].is_unique)
print('Số dòng trùng toàn bộ:', promotions.duplicated().sum())

promo_id là unique: True
Số dòng trùng toàn bộ: 0


In [5]:
# Chuyển cột ngày sang datetime
promotions['start_date'] = pd.to_datetime(promotions['start_date'], errors='coerce')
promotions['end_date'] = pd.to_datetime(promotions['end_date'], errors='coerce')
print('Số start_date không đọc được:', promotions['start_date'].isna().sum())
print('Số end_date không đọc được:', promotions['end_date'].isna().sum())

Số start_date không đọc được: 0
Số end_date không đọc được: 0


In [6]:
# Xem các giá trị promo_type
promotions['promo_type'].value_counts()

promo_type
percentage    45
fixed          5
Name: count, dtype: int64

In [7]:
# Xem applicable_category null nghĩa là áp dụng cho tất cả category theo data dictionary
display(promotions['applicable_category'].value_counts(dropna=False))
# Thay thế tất cả Nan bằng All
promotions['applicable_category'] = promotions['applicable_category'].fillna("All")
# Kiểm tra
display(promotions['applicable_category'].value_counts(dropna=False))


applicable_category
NaN           40
Streetwear     5
Outdoor        5
Name: count, dtype: int64

applicable_category
All           40
Streetwear     5
Outdoor        5
Name: count, dtype: int64

In [8]:
# Xem promo_channel
promotions['promo_channel'].value_counts()

promo_channel
all_channels    19
online          13
email            7
social_media     6
in_store         5
Name: count, dtype: int64

In [9]:
# Xem stackable_flag (0 hoặc 1)
promotions['stackable_flag'].value_counts()

stackable_flag
0    38
1    12
Name: count, dtype: int64

In [10]:
# Đối chiếu master data: applicable_category phải khớp với category trong bảng products
products = pd.read_csv('../products.csv')
valid_cats = set(products['category'].unique())
actual_cats = set(promotions['applicable_category'].dropna().unique())
print('Category hợp lệ (products):', sorted(valid_cats))
print('Category trong promotions  :', sorted(actual_cats))
print('Category không hợp lệ      :', actual_cats - valid_cats)
# Không cần quan tâm All vì ta đã chuẩn hóa nó trước đó.

Category hợp lệ (products): ['Casual', 'GenZ', 'Outdoor', 'Streetwear']
Category trong promotions  : ['All', 'Outdoor', 'Streetwear']
Category không hợp lệ      : {'All'}


In [11]:
# Chuyển kiểu dữ liệu
promotions['promo_type'] = promotions['promo_type'].astype('category')
promotions['applicable_category'] = promotions['applicable_category'].astype('category')
promotions['promo_channel'] = promotions['promo_channel'].astype('category')
promotions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   promo_id             50 non-null     object        
 1   promo_name           50 non-null     object        
 2   promo_type           50 non-null     category      
 3   discount_value       50 non-null     float64       
 4   start_date           50 non-null     datetime64[ns]
 5   end_date             50 non-null     datetime64[ns]
 6   applicable_category  50 non-null     category      
 7   promo_channel        50 non-null     category      
 8   stackable_flag       50 non-null     int64         
 9   min_order_value      50 non-null     int64         
dtypes: category(3), datetime64[ns](2), float64(1), int64(2), object(2)
memory usage: 3.5+ KB


In [12]:
# Tổng kết promotions
print('Tổng số dòng     :', len(promotions))
print('Tổng số null     :', promotions.isna().sum())

Tổng số dòng     : 50
Tổng số null     : promo_id               0
promo_name             0
promo_type             0
discount_value         0
start_date             0
end_date               0
applicable_category    0
promo_channel          0
stackable_flag         0
min_order_value        0
dtype: int64


## Phần 2: Đọc và làm sạch EPROM

In [13]:
eprom = pd.read_json('../eprom.json', convert_dates=False)
eprom_raw = eprom.copy()
display(eprom.head())
eprom.info()

,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
0,PROMO-0039-0001,Fall Launch 2020,percentage,10.0,2020-08-30,2020-10-01,None,all_channels,0,150000
1,PROMO-0029-0002,Fall Launch 2018,percentage,10.0,2018-08-30,2018-10-01,None,email,0,0
2,PROMO-0015-0003,Urban Blowout 2015,fixed,50.0,2015-07-30,2015-09-02,Streetwear,online,0,200000
3,PROMO-0043-0004,Fall Launch 2021,percentage,10.0,2021-08-30,2021-10-02,None,email,0,0
4,PROMO-0008-0005,Mid-Year Sale 2014,percentage,18.0,2014-06-23,2014-07-22,None,social_media,0,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   promo_id             1000 non-null   object
 1   promo_name           999 non-null    object
 2   promo_type           1000 non-null   object
 3   discount_value       1000 non-null   object
 4   start_date           1000 non-null   object
 5   end_date             1000 non-null   object
 6   applicable_category  217 non-null    object
 7   promo_channel        1000 non-null   object
 8   stackable_flag       1000 non-null   int64 
 9   min_order_value      1000 non-null   int64 
dtypes: int64(2), object(8)
memory usage: 78.3+ KB


In [14]:
# Chuẩn hóa khoảng trắng ở cả hai bảng
text_cols = ['promo_id', 'promo_name', 'promo_type',
             'applicable_category', 'promo_channel']
for col in text_cols:
    promotions[col] = promotions[col].astype('string').str.strip().replace('', pd.NA)
    eprom[col] = eprom[col].astype('string').str.strip().replace('', pd.NA)

# Category thiếu nghĩa là áp dụng cho tất cả, theo Dictionary
promotions['applicable_category'] = promotions['applicable_category'].fillna('All')
eprom['applicable_category'] = eprom['applicable_category'].fillna('All')
eprom['promo_channel'] = eprom['promo_channel'].replace(['N/A', 'n/a'], pd.NA)

In [15]:
# Lấy mã khuyến mãi gốc: PROMO-0001-0020 -> PROMO-0001
# Chỉ chấp nhận ID đúng cấu trúc
eprom['master_promo_id'] = eprom['promo_id'].str.extract(
    r'^(PROMO-\d{4})-\d{4}$', expand=False
)
display(eprom[eprom['master_promo_id'].isna()])
display(eprom[eprom['promo_id'].duplicated(keep=False)])

# Dùng tên master_promo_id ở bảng tham chiếu để merge cho dễ
reference = promotions.rename(columns={'promo_id': 'master_promo_id'}).copy()
print('Mã gốc chưa có trong Promotions:')
display(eprom[~eprom['master_promo_id'].isin(reference['master_promo_id'])])

,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value,master_promo_id


,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value,master_promo_id


Mã gốc chưa có trong Promotions:


,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value,master_promo_id


In [16]:
# Chuẩn hóa ngày và số; chuỗi không đọc được sẽ thành null
for col in ['start_date', 'end_date']:
    eprom[col] = pd.to_datetime(eprom[col], format='%Y-%m-%d', errors='coerce')

number_cols = ['discount_value', 'stackable_flag', 'min_order_value']
for col in number_cols:
    eprom[col] = pd.to_numeric(eprom[col], errors='coerce')
    reference[col] = pd.to_numeric(reference[col], errors='coerce')

eprom[number_cols] = eprom[number_cols].replace([np.inf, -np.inf], np.nan)
eprom[['discount_value', 'min_order_value']] = eprom[['discount_value', 'min_order_value']].round(6)
reference[['discount_value', 'min_order_value']] = reference[['discount_value', 'min_order_value']].round(6)
print(eprom.isna().sum())

promo_id               0
promo_name             2
promo_type             0
discount_value         2
start_date             0
end_date               0
applicable_category    0
promo_channel          1
stackable_flag         0
min_order_value        0
master_promo_id        0
dtype: int64


### Tên khuyến mãi bị thiếu

In [17]:
missing = eprom[eprom['promo_name'].isna()].reset_index()
display(missing)

,index,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value,master_promo_id
0,17,PROMO-0002-0018,<NA>,percentage,18.0,2013-06-23,2013-07-22,All,online,0,0,PROMO-0002
1,286,PROMO-0009-0287,<NA>,percentage,10.0,2014-08-30,2014-10-01,All,all_channels,0,100000,PROMO-0009


In [18]:
# So sánh mã gốc và các trường còn lại, không dùng trường đang lỗi
cols = ['master_promo_id', 'promo_type', 'discount_value', 'start_date', 'end_date', 'applicable_category', 'promo_channel', 'stackable_flag', 'min_order_value']
matched = missing.merge(reference[cols].drop_duplicates(), on=cols, how='inner')
display(matched)

# Chỉ drop dòng lỗi đã có bản tương ứng trong Promotions
print('Số dòng được bỏ:', matched['index'].nunique())
eprom = eprom.drop(index=matched['index'])

,index,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value,master_promo_id
0,17,PROMO-0002-0018,<NA>,percentage,18.0,2013-06-23,2013-07-22,All,online,0,0,PROMO-0002
1,286,PROMO-0009-0287,<NA>,percentage,10.0,2014-08-30,2014-10-01,All,all_channels,0,100000,PROMO-0009


Số dòng được bỏ: 2


### Mức giảm bị thiếu hoặc không đọc được

In [19]:
missing = eprom[eprom['discount_value'].isna()].reset_index()
display(missing)

,index,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value,master_promo_id
0,73,PROMO-0034-0074,Year-End Sale 2019,percentage,NaN,2019-11-18,2020-01-02,All,all_channels,0,50000,PROMO-0034
1,514,PROMO-0019-0515,Fall Launch 2016,percentage,NaN,2016-08-30,2016-10-01,All,online,0,0,PROMO-0019


In [20]:
# So sánh mã gốc và các trường còn lại, không dùng trường đang lỗi
cols = ['master_promo_id', 'promo_name', 'promo_type', 'start_date', 'end_date', 'applicable_category', 'promo_channel', 'stackable_flag', 'min_order_value']
matched = missing.merge(reference[cols].drop_duplicates(), on=cols, how='inner')
display(matched)

# Chỉ drop dòng lỗi đã có bản tương ứng trong Promotions
print('Số dòng được bỏ:', matched['index'].nunique())
eprom = eprom.drop(index=matched['index'])

,index,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value,master_promo_id
0,73,PROMO-0034-0074,Year-End Sale 2019,percentage,NaN,2019-11-18,2020-01-02,All,all_channels,0,50000,PROMO-0034
1,514,PROMO-0019-0515,Fall Launch 2016,percentage,NaN,2016-08-30,2016-10-01,All,online,0,0,PROMO-0019


Số dòng được bỏ: 2


### Mức giảm âm hoặc phần trăm lớn hơn 100

In [21]:
missing = eprom[(eprom['discount_value'] < 0) | ((eprom['promo_type'] == 'percentage') & (eprom['discount_value'] > 100))].reset_index()
display(missing)

,index,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value,master_promo_id
0,121,PROMO-0034-0122,Year-End Sale 2019,percentage,150.0,2019-11-18,2020-01-02,All,all_channels,0,50000,PROMO-0034
1,622,PROMO-0012-0623,Mid-Year Sale 2015,percentage,-10.0,2015-06-23,2015-07-22,All,social_media,0,0,PROMO-0012


In [22]:
# So sánh mã gốc và các trường còn lại, không dùng trường đang lỗi
cols = ['master_promo_id', 'promo_name', 'promo_type', 'start_date', 'end_date', 'applicable_category', 'promo_channel', 'stackable_flag', 'min_order_value']
matched = missing.merge(reference[cols].drop_duplicates(), on=cols, how='inner')
display(matched)

# Chỉ drop dòng lỗi đã có bản tương ứng trong Promotions
print('Số dòng được bỏ:', matched['index'].nunique())
eprom = eprom.drop(index=matched['index'])

,index,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value,master_promo_id
0,121,PROMO-0034-0122,Year-End Sale 2019,percentage,150.0,2019-11-18,2020-01-02,All,all_channels,0,50000,PROMO-0034
1,622,PROMO-0012-0623,Mid-Year Sale 2015,percentage,-10.0,2015-06-23,2015-07-22,All,social_media,0,0,PROMO-0012


Số dòng được bỏ: 2


### Ngày bị lỗi hoặc ngày kết thúc trước ngày bắt đầu

In [23]:
missing = eprom[eprom['start_date'].isna() | eprom['end_date'].isna() | (eprom['end_date'] < eprom['start_date'])].reset_index()
display(missing)

,index,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value,master_promo_id
0,333,PROMO-0022-0334,Mid-Year Sale 2017,percentage,18.0,2026-12-31,2026-01-01,All,online,0,150000,PROMO-0022


In [24]:
# So sánh mã gốc và các trường còn lại, không dùng trường đang lỗi
cols = ['master_promo_id', 'promo_name', 'promo_type', 'discount_value', 'applicable_category', 'promo_channel', 'stackable_flag', 'min_order_value']
matched = missing.merge(reference[cols].drop_duplicates(), on=cols, how='inner')
display(matched)

# Chỉ drop dòng lỗi đã có bản tương ứng trong Promotions
print('Số dòng được bỏ:', matched['index'].nunique())
eprom = eprom.drop(index=matched['index'])

,index,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value,master_promo_id
0,333,PROMO-0022-0334,Mid-Year Sale 2017,percentage,18.0,2026-12-31,2026-01-01,All,online,0,150000,PROMO-0022


Số dòng được bỏ: 1


### Kênh khuyến mãi bị thiếu hoặc không hợp lệ

In [25]:
missing = eprom[~eprom['promo_channel'].isin(['all_channels', 'online', 'email', 'social_media', 'in_store'])].reset_index()
display(missing)

,index,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value,master_promo_id
0,845,PROMO-0038-0846,Mid-Year Sale 2020,percentage,18.0,2020-06-23,2020-07-22,All,<NA>,0,0,PROMO-0038


In [26]:
# So sánh mã gốc và các trường còn lại, không dùng trường đang lỗi
cols = ['master_promo_id', 'promo_name', 'promo_type', 'discount_value', 'start_date', 'end_date', 'applicable_category', 'stackable_flag', 'min_order_value']
matched = missing.merge(reference[cols].drop_duplicates(), on=cols, how='inner')
display(matched)

# Chỉ drop dòng lỗi đã có bản tương ứng trong Promotions
print('Số dòng được bỏ:', matched['index'].nunique())
eprom = eprom.drop(index=matched['index'])

,index,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value,master_promo_id
0,845,PROMO-0038-0846,Mid-Year Sale 2020,percentage,18.0,2020-06-23,2020-07-22,All,<NA>,0,0,PROMO-0038


Số dòng được bỏ: 1


## Phần 3: Đối chiếu tất cả thông tin và chỉ thêm mã mới

In [27]:
# Không so sánh ID biến thể; dùng mã gốc và tất cả thông tin khuyến mãi
compare_cols = ['master_promo_id', 'promo_name', 'promo_type', 'discount_value',
                'start_date', 'end_date', 'applicable_category', 'promo_channel',
                'stackable_flag', 'min_order_value']
merged = eprom.merge(
    reference[compare_cols].drop_duplicates(),
    on=compare_cols, how='left', indicator=True
)
already_exists = merged[merged['_merge'] == 'both']
not_matched = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')
print('EPROM đã có và khớp toàn bộ:', len(already_exists))
print('EPROM chưa khớp:', len(not_matched))
display(not_matched)

EPROM đã có và khớp toàn bộ: 992
EPROM chưa khớp: 0


,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value,master_promo_id


In [28]:
# Cùng mã gốc nhưng khác thông tin: giữ Promotions, xem riêng các khác biệt
conflicts = not_matched[not_matched['master_promo_id'].isin(promotions['promo_id'])].copy()
display(conflicts.merge(reference, on='master_promo_id', suffixes=('_eprom', '_promotions')))

# Chỉ lấy mã gốc chưa có trong Promotions
new_from_eprom = not_matched[~not_matched['master_promo_id'].isin(promotions['promo_id'])].copy()

# Silver dùng mã gốc, không dùng mã biến thể
new_from_eprom = new_from_eprom.drop(columns='promo_id')
new_from_eprom = new_from_eprom.rename(columns={'master_promo_id': 'promo_id'})
new_from_eprom = new_from_eprom[promotions.columns].drop_duplicates()

# Một mã mới có nhiều thông tin khác nhau thì giữ riêng để kiểm tra
new_id_conflicts = new_from_eprom[new_from_eprom['promo_id'].duplicated(keep=False)].copy()
display(new_id_conflicts)
new_from_eprom = new_from_eprom.drop(index=new_id_conflicts.index)
print('Khuyến mãi mới được thêm:', len(new_from_eprom))
display(new_from_eprom)

,promo_id,promo_name_eprom,promo_type_eprom,discount_value_eprom,start_date_eprom,end_date_eprom,applicable_category_eprom,promo_channel_eprom,stackable_flag_eprom,min_order_value_eprom,master_promo_id,promo_name_promotions,promo_type_promotions,discount_value_promotions,start_date_promotions,end_date_promotions,applicable_category_promotions,promo_channel_promotions,stackable_flag_promotions,min_order_value_promotions


,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value


Khuyến mãi mới được thêm: 0


,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value


In [29]:
# Gộp và kiểm tra trước khi xuất
promotions_silver = pd.concat([promotions, new_from_eprom], ignore_index=True)
promotions_silver = promotions_silver.sort_values('promo_id').reset_index(drop=True)
promotions_silver['stackable_flag'] = promotions_silver['stackable_flag'].astype('Int64')

print('Tổng số dòng:', len(promotions_silver))
print('Số mã trùng:', promotions_silver['promo_id'].duplicated().sum())
print('Số null:', promotions_silver.isna().sum().sum())
promotions_silver.info()

Tổng số dòng: 50
Số mã trùng: 0
Số null: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   promo_id             50 non-null     string        
 1   promo_name           50 non-null     string        
 2   promo_type           50 non-null     string        
 3   discount_value       50 non-null     float64       
 4   start_date           50 non-null     datetime64[ns]
 5   end_date             50 non-null     datetime64[ns]
 6   applicable_category  50 non-null     string        
 7   promo_channel        50 non-null     string        
 8   stackable_flag       50 non-null     Int64         
 9   min_order_value      50 non-null     int64         
dtypes: Int64(1), datetime64[ns](2), float64(1), int64(1), string(5)
memory usage: 4.1 KB


In [30]:

promotions_silver.to_csv('../SilverData/promotions_silver.csv', index=False)
print('Đã xuất promotions_silver.csv')

Đã xuất promotions_silver.csv
